# HeritageGuard — Pelatihan Model ML (Google Colab)

Notebook ini melatih dua model HeritageGuard di **Google Colab** (gunakan GPU gratis):

1. **NMT (terjemahan)** — fine-tuning `facebook/nllb-200-distilled-600M` untuk Indonesia ↔ Jawa ↔ Madura dengan kontrol tingkat tutur (halus/lugu).
2. **Classifier (deteksi bahasa & register)** — TF-IDF + Logistic Regression (scikit-learn).

## Cara pakai
1. Buka di Colab → menu **Runtime ▸ Change runtime type ▸ GPU (T4)**.
2. Jalankan sel berurutan dari atas.
3. Di akhir, unduh `heritageguard_models.zip` lalu ekstrak ke folder `models/` di root repo.

> Catatan: dataset proyek kecil, jadi ekspektasi kualitas terjemahan realistis. Backend tetap punya fallback rule-based bila artefak tidak ada.

## 1. Pasang dependensi

In [ ]:
!pip -q install "transformers>=4.40" "datasets>=2.18" "sacrebleu>=2.4" "scikit-learn>=1.3" sentencepiece accelerate joblib
import torch
print('CUDA tersedia:', torch.cuda.is_available())

## 2. Ambil kode + dataset

Pilih salah satu: **(A)** clone dari GitHub repo kamu, atau **(B)** upload manual folder `Dataset/` dan `backend/`.

Ganti URL di bawah dengan repo kamu.

In [ ]:
import os
REPO_URL = 'https://github.com/USERNAME/FP-AI-Kelompok7.git'  # <-- GANTI
if not os.path.exists('FP-AI-Kelompok7'):
    !git clone $REPO_URL
%cd FP-AI-Kelompok7
!ls Dataset

## 3. Siapkan korpus dari dataset lokal

Memakai modul `backend.ml.data_prep` (tanpa dependensi berat).

In [ ]:
from backend.ml.data_prep import write_corpora
from backend.ml import config
stats = write_corpora()
stats

In [ ]:
import json
parallel = [json.loads(l) for l in open(config.PARALLEL_CORPUS_PATH, encoding='utf-8')]
clf_rows = [json.loads(l) for l in open(config.CLASSIFIER_CORPUS_PATH, encoding='utf-8')]
print('parallel pairs :', len(parallel))
print('classifier rows:', len(clf_rows))
parallel[:3]

## 4. Latih classifier (TF-IDF + Logistic Regression)

Menghasilkan bundle dengan: `lang_clf` (bahasa), `register_clf` (register per bahasa), `style_clf` (high/low untuk persentase ngoko-krama).

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
import joblib, collections

def make_pipeline():
    return Pipeline([
        ('tfidf', TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=2)),
        ('clf', LogisticRegression(max_iter=1000, C=4.0)),
    ])

# --- 4a. Language classifier ---
texts = [r['text'] for r in clf_rows]
langs = [r['language'] for r in clf_rows]
Xtr, Xte, ytr, yte = train_test_split(texts, langs, test_size=0.15, random_state=42, stratify=langs)
lang_clf = make_pipeline().fit(Xtr, ytr)
print('=== LANGUAGE ===')
print(classification_report(yte, lang_clf.predict(Xte)))

In [ ]:
# --- 4b. Style classifier (high/low) per regional language, for ngoko/krama % ---
# Build from the parallel corpus: regional-side text labelled by its level.
style_data = collections.defaultdict(lambda: ([], []))
for r in parallel:
    for side_lang, side_text in ((r['src_lang'], r['source']), (r['tgt_lang'], r['target'])):
        if side_lang == 'jv':
            style_data['Jawa'][0].append(side_text); style_data['Jawa'][1].append(r['level'])
        elif side_lang == 'mad':
            style_data['Madura'][0].append(side_text); style_data['Madura'][1].append(r['level'])

style_clf = {}
register_clf = {}
for lang_label, (X, y) in style_data.items():
    if len(set(y)) < 2:
        continue
    sc = make_pipeline().fit(X, y)
    style_clf[lang_label] = sc
    # Register label derived from style for the detector endpoint.
    if lang_label == 'Jawa':
        reg_map = {'high': 'krama alus', 'low': 'ngoko lugu'}
    else:
        reg_map = {'high': 'Engghi-bhunten', 'low': 'Enja-Iya'}
    yr = [reg_map[v] for v in y]
    register_clf[lang_label] = make_pipeline().fit(X, yr)
    print(lang_label, 'style classes:', list(sc.classes_))

# Indonesia register (formal/informal) using slang word heuristic as labels.
indo_slang = {'gue','gua','lu','nggak','aja','udah','lagi','kenapa','banget','pengen','bobo','mager','bodo','yuk','bro','dong','capek'}
indo_texts = [r['text'] for r in clf_rows if r['language'] == 'Indonesia']
indo_labels = ['informal' if any(w in t.lower().split() for w in indo_slang) else 'formal' for t in indo_texts]
if len(set(indo_labels)) == 2:
    register_clf['Indonesia'] = make_pipeline().fit(indo_texts, indo_labels)
    print('Indonesia register classes:', list(register_clf['Indonesia'].classes_))

In [ ]:
bundle = {'lang_clf': lang_clf, 'register_clf': register_clf, 'style_clf': style_clf}
os.makedirs(config.MODEL_DIR, exist_ok=True)
joblib.dump(bundle, config.CLASSIFIER_PATH)
print('Saved classifier ->', config.CLASSIFIER_PATH)

## 5. Fine-tune NMT (NLLB-200 distilled)

Sumber diberi tag tingkat tutur `<halus>`/`<lugu>` agar model bisa dikendalikan register-nya. Madura memakai kode proxy `ind_Latn` lalu diadaptasi lewat fine-tuning.

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from datasets import Dataset

BASE_MODEL = config.BASE_NMT_MODEL
LEVEL_TAGS = config.LEVEL_TAGS
NLLB = config.NLLB_LANG_CODES

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE_MODEL)

def to_example(r):
    return {
        'src_text': f"{LEVEL_TAGS.get(r['level'],'')} {r['source']}".strip(),
        'tgt_text': r['target'],
        'src_code': NLLB[r['src_lang']],
        'tgt_code': NLLB[r['tgt_lang']],
    }

examples = [to_example(r) for r in parallel]
ds = Dataset.from_list(examples).train_test_split(test_size=0.05, seed=42)
print(ds)

In [ ]:
MAX_LEN = 96
def preprocess(batch):
    # NLLB encodes source under its src_lang; use the first row's code per batch.
    tok.src_lang = batch['src_code'][0]
    enc = tok(batch['src_text'], max_length=MAX_LEN, truncation=True, padding='max_length')
    labels = tok(text_target=batch['tgt_text'], max_length=MAX_LEN, truncation=True, padding='max_length')
    enc['labels'] = labels['input_ids']
    return enc

tokenized = ds.map(preprocess, batched=True, batch_size=64, remove_columns=ds['train'].column_names)
tokenized

In [ ]:
from transformers import DataCollatorForSeq2Seq, Seq2SeqTrainer, Seq2SeqTrainingArguments

collator = DataCollatorForSeq2Seq(tok, model=model)
args = Seq2SeqTrainingArguments(
    output_dir='nmt_ckpt',
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=5e-5,
    num_train_epochs=3,
    weight_decay=0.01,
    fp16=torch.cuda.is_available(),
    logging_steps=100,
    save_strategy='no',
    report_to='none',
)
trainer = Seq2SeqTrainer(
    model=model, args=args,
    train_dataset=tokenized['train'], eval_dataset=tokenized['test'],
    data_collator=collator, tokenizer=tok,
)
trainer.train()

In [ ]:
os.makedirs(config.NMT_DIR, exist_ok=True)
trainer.save_model(str(config.NMT_DIR))
tok.save_pretrained(str(config.NMT_DIR))
print('Saved NMT ->', config.NMT_DIR)

## 6. Evaluasi cepat (BLEU) + uji manual

In [ ]:
import sacrebleu
def translate(text, src, tgt, level='high'):
    tok.src_lang = NLLB[src]
    inp = tok(f"{LEVEL_TAGS.get(level,'')} {text}".strip(), return_tensors='pt', truncation=True, max_length=MAX_LEN).to(model.device)
    out = model.generate(**inp, forced_bos_token_id=tok.convert_tokens_to_ids(NLLB[tgt]), max_length=MAX_LEN, num_beams=4)
    return tok.batch_decode(out, skip_special_tokens=True)[0]

for t in ['Saya ingin makan nasi goreng.', 'Kamu mau pergi ke mana?', 'Terima kasih banyak.']:
    print(t, '->', translate(t, 'id', 'jv', 'high'))

## 7. Bundel & unduh artefak

In [ ]:
import shutil
shutil.make_archive('heritageguard_models', 'zip', str(config.MODEL_DIR))
print('Zip dibuat: heritageguard_models.zip')
try:
    from google.colab import files
    files.download('heritageguard_models.zip')
except Exception as e:
    print('Unduh manual dari panel Files. Detail:', e)

## 8. Pasang di backend

Ekstrak `heritageguard_models.zip` ke folder `models/` di root repo sehingga strukturnya:

```
models/
  nmt/                 # model + tokenizer hasil fine-tune
  register_clf.joblib  # bundle classifier
```

Lalu jalankan backend seperti biasa. `backend/ml/inference.py` akan otomatis memuat artefak ini; jika tidak ada, backend tetap jalan dengan fallback rule-based.